# Train Logistic Classifier

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [8]:
def load_classifier_data(file_path):

    # load NK + SK from test set
    df = pd.read_csv(file_path, sep="\t")
    # NK sentences (input side)
    nk_sentences = df["nk"].astype(str).tolist()
    # SK sentences (reference side)
    sk_sentences = df["sk"].astype(str).tolist()

    # create labels: NK=0, SK=1
    texts = nk_sentences + sk_sentences
    labels = [0] * len(nk_sentences) + [1] * len(sk_sentences)
    return texts, labels

In [9]:
# train classifier
def train_classifier(texts, labels):

    X_train, X_eval, y_train, y_eval = train_test_split(
        # 80% are used to train classifer, 20% are used to evaluate classifier
        texts, labels, test_size=0.2, random_state=42, stratify=labels
    )

    model = Pipeline([
        # transforms sentence into numeric vector
        ("vectorizer", CountVectorizer(analyzer="char", ngram_range=(1, 3))),
        ("clf", LogisticRegression(max_iter=1000))
    ])

    # train
    model.fit(X_train, y_train)

    # evaluate
    preds = model.predict(X_eval)
    acc = accuracy_score(y_eval, preds)
    print(f"\nClassifier Accuracy: {acc:.4f}")
    print(classification_report(y_eval, preds, target_names=["NK", "SK"]))
    return model

Plain Text : 그는 생각했다. <br>
Character-level view : 그/는/ /생/각/했/다/ . <br>

1-grams <br>
그, 는,  , 생, 각, 했, 다, . <br><br>

2-grams <br>
그는,는 , 생,생각,각했,했다,다. <br><br>

3-grams <br>
그는 ,는 생, 생각,생각했,각했다,했다. <br><br>

CountVectorizer builds a huge vocabulary like: ["그", "는", "생", "각", "그는", "생각", "각했", "했다", ...] then represents the sentence as a vector = [count of each n-gram] which becomes an input to logistic regression.

The model is not reading meaning but rather learning common character patterns each in NK and SK so that it can detect whether a sentence is written in South Korean or North Korean.

# Evaluate each model

In [10]:
def evaluate_outputs(model, file_path, save_path):

    # load output sentences from each model
    df = pd.read_csv(file_path, sep="\t")
    # model-generated translations (NK -> SK)
    hyp = df["hyp_sk"].astype(str).tolist()
    
    # how confident is the classifier that this sentence is SK?
    probs = model.predict_proba(hyp)[:, 1]  # P(SK)
    # predict_proba returns for each sentence, [P(NK), P(SK)]
    # [:, 1] : return only the probability of the sentence being SK
    
    # if P(SK) > 0.5, classify as SK (1)
    preds = (probs > 0.5).astype(int)
    
    # create column in dataframe
    df["P(SK)"] = probs
    df["pred_label"] = preds  # 1=SK, 0=NK
    
    # percentage classified as SK (binary classification)
    percent_sk = np.mean(preds) * 100
    
    # on average, outputs are ?% likely to be SK (how strongly SK-like)
    avg_prob = np.mean(probs)
    
    print(f"\nEvaluation on: {file_path}")
    print(f"% classified as SK: {percent_sk:.2f}%")
    print(f"Average P(SK): {avg_prob:.4f}")

    # save full dataframe
    df.to_csv(save_path, sep="\t", index=False)
    
    print(f"Saved detailed results to: {save_path}")
    
    return df, percent_sk, avg_prob

In [11]:
if __name__ == "__main__":

    # train classifier
    
    results = []
    test_file = "../data/bilingual/filtered/test_filtered.tsv"
    
    texts, labels = load_classifier_data(test_file)
    model = train_classifier(texts, labels)
    
    # evaluate each model
    results = []
    
    models = [
        ("baseline", "../data/translation_results/baseline_nk_to_sk_test.tsv"),
        ("1.0x", "../data/translation_results/aug_1.0x_nk_to_sk_test.tsv"),
        ("2.0x", "../data/translation_results/aug_2.0x_nk_to_sk_test.tsv"),
        ("3.0x", "../data/translation_results/aug_3.0x_nk_to_sk_test.tsv"),
        ("4.0x", "../data/translation_results/aug_4.0x_nk_to_sk_test.tsv"),
    ]

    for name, path in models:
        save_path = f"../data/classifier_results/{name}_classifier_scores.tsv"
        df, percent_sk, avg_prob = evaluate_outputs(model, path, save_path)
        results.append((name, percent_sk, avg_prob))

    # save summary
    summary_df = pd.DataFrame(results, columns=["model", "%_SK", "avg_P(SK)"])
    summary_df.to_csv("../data/classifier_results/classifier_summary.csv", index=False)
    print("\nSummary:")
    print(summary_df)


Classifier Accuracy: 0.9167
              precision    recall  f1-score   support

          NK       0.90      0.94      0.92      1741
          SK       0.93      0.90      0.92      1741

    accuracy                           0.92      3482
   macro avg       0.92      0.92      0.92      3482
weighted avg       0.92      0.92      0.92      3482


Evaluation on: ../data/translation_results/baseline_nk_to_sk_test.tsv
% classified as SK: 84.56%
Average P(SK): 0.8289
Saved detailed results to: ../data/classifier_results/baseline_classifier_scores.tsv

Evaluation on: ../data/translation_results/aug_1.0x_nk_to_sk_test.tsv
% classified as SK: 85.71%
Average P(SK): 0.8356
Saved detailed results to: ../data/classifier_results/1.0x_classifier_scores.tsv

Evaluation on: ../data/translation_results/aug_2.0x_nk_to_sk_test.tsv
% classified as SK: 85.19%
Average P(SK): 0.8351
Saved detailed results to: ../data/classifier_results/2.0x_classifier_scores.tsv

Evaluation on: ../data/translation_r

The logistic classifier results show that all models consistently generate outputs that are classified as South Korean at a high rate (approximately 84–86%), indicating that the baseline model is already strong.

Increasing the amount of synthetic data does not lead to meaningful improvement. While a slight increase is observed at lower augmentation scales (1.0×–3.0×), the performance quickly plateaus and even slightly decreases at higher scales (4.0×).

This indicates that back-translation does not significantly enhance dialect-specific transformation. 

One possible explanation is that the synthetic data, generated by the reverse SK->NK model, does not introduce sufficiently diverse or novel dialectal features beyond what is already captured in the original parallel corpus.

Another explanation is that the classifier, based on character n-gram, captures surface-level orthographic patterns, and may therefore fail to detect possible improvements in higher-level semantic, syntactic or stylistic transformations.

# Extract interesting sentence pairs

Analyze the baseline model

In [12]:
df = pd.read_csv("../data/classifier_results/baseline_classifier_scores.tsv", sep="\t")

High-confidence SK

In [21]:
high_conf = df.sort_values(by="P(SK)", ascending=False).head(20)

high_conf.to_csv("../data/classifier_results/baseline_high_confidence.tsv", sep="\t", index=False)

print("\nTop high-confidence SK:")
# print(high_conf[["nk", "ref_sk", "hyp_sk", "P(SK)"]])


Top high-confidence SK:


In [22]:
print(high_conf[["hyp_sk", "P(SK)"]])

                                                 hyp_sk  P(SK)
1003  그것은 로체스터 씨가 살아 있다는 것을 알고 있었기 때문이고, 또한 인간으로서, 더...    1.0
5670  잉그램 경은 옘미 이쉬튼 양과 농담을 하고 있었고, 루이자 양은 피아노를 치거나 린...    1.0
5814  내가 여러분에게 갈 준비를 한 것은 이번이 세 번째입니다. 여러분에게 짐이 되지는 ...    1.0
7890  다윗이 자기 아들 솔로몬에게 말했습니다. 여호와의 성전에서 쓸 모든 것을 만들 때까...    1.0
4413  사울이 대답했습니다. 그것은 아말렉 사람들이 빼앗아 온 것입니다. 사울은 양 떼와 ...    1.0
3795  그들은 여호와께 태워 드리는 제물인 번제물로 일 년 된 흠 없는 숫양 한 마리를 바...    1.0
1695  여러분의 하나님 여호와께서 여러분에게 하시는 모든 일 곧 여러분의 몸과 여러분의 짐...    1.0
1104  북쪽 문에서 사람 여섯 명이 나왔습니다. 그들은 모두 손에 망치를 들고 있었습니다....    1.0
5709  내 백성이 나를 알지 못하여 망할 것이다. 너희가 제사장이라는 것을 내게 알리지 않...    1.0
1875  그 날에 너의 입은 활짝 열려 너희가 도망치는 자에게 말을 할 수 있게 될 것이며 ...    1.0
1567  쥘리앵이 자신의 지식과 베리에르에 대한 추억으로 만들어 낸 추억으로 사랑하는 사람을...    1.0
6987  여호와께서 그 영에게 물으셨습니다. 어떻게 그를 꾀어 낼 수 있겠느냐? 그 영이 대...    1.0
1203  소인국이라든가, 대인국이라든가는 내가 믿고 있는 대로 지구에 속해 있는 육지에 속해...    1.0
1514  소인국이라든가, 대인국이라든가는 내가 믿고 있는 대로 지구에 속해 있는 육지에 속해...    1.0
5542  난롯가에 있는 내 방으로 날라다 주어야 한다는 것과, 고기와 버터를 바른 빵이 들어...

low-confidence SK

In [23]:
low_conf = df.sort_values(by="P(SK)").head(20)

low_conf.to_csv("../data/classifier_results/baseline_low_confidence.tsv", sep="\t", index=False)

print("\nTop low-confidence SK:")
# print(low_conf[["nk", "ref_sk", "hyp_sk", "P(SK)"]])


Top low-confidence SK:


In [24]:
print(low_conf[["hyp_sk", "P(SK)"]])

                                                 hyp_sk     P(SK)
429   여호와께서 이렇게 말씀하셨다. 보아라. 내가 그에게 재앙을 내리겠다. 주 여호와께서...  0.000014
6877               건장한 종은 머리를 조아리며 말하였다. 소졸 이름은 신비롭소이다.  0.000326
1511               어찌하여 나라들이 술렁거리는가 어찌하여 민족들이 헛일을 꾸미는가?  0.000709
1734  심청이가 들어보니 분명히 자기가 죽을 꿈이므로 마음속이 섬찍하였다. 그러나 겉으로는...  0.001471
7313  아버지께서 제 몸을 만지시면 제가 아버지께 죄를 지었나이다. 제가 아버지께 죄를 지...  0.003171
3624  그들 가운데 누가 여호와의 회의에 참석하여 그분을 뵈었느냐? 누가 그분의 말씀에 귀...  0.003426
4031                              기피한 사랑이라니, 기피한 사랑이라니!  0.005824
6818                              기피한 사랑이라니, 기피한 사랑이라니!  0.005824
4941                              기피한 사랑이라니, 기피한 사랑이라니!  0.005824
7293  그러나 우리가 그들의 비위를 건드릴 필요는 없소. 먼저 바다에 가서 먼저 잡은 고기...  0.005895
292   음모를 꾸미고 모여들어 화덕처럼 내 마음에 불을 지피고 밤새도록 타오르는 가슴을 진...  0.006314
3091                게다가 이 새로운 방식은 상대방에게 일종의 흥미를 불러일으켰다.  0.009992
8388                게다가 이 새로운 방식은 상대방에게 일종의 흥미를 불러일으켰다.  0.009992
3770  또한 장원의 풍채 끼끗하면서도 시원스럽게 름름하고 기품이 헌앙하여 만인 중에 뛰어나...  0.010832
7320      

Analysis of the results to be added